# Qdrant Vector Storage

## Overview

This document explains how two classes work together to form the vector persistence layer: `QdrantStore` (for the ingestion service — upserts vectors with metadata) and `QdrantRetriever` (for the chat service — searches for similar vectors). Together they're the persistence layer for the RAG pipeline.

A vector database is specialized for one job: finding the N most similar vectors to a query vector, fast. Regular databases (Postgres, MongoDB) can store vectors, but they can't do efficient similarity search at scale. Qdrant is purpose-built for this.

## Architecture Context

- **Ingestion:** After embedding text chunks, `QdrantStore.upsert()` stores the vectors + metadata (filename, page number, chunk text) in Qdrant
- **Chat:** When a user asks a question, `QdrantRetriever.search()` finds the top-K most similar chunks and returns them with their metadata for building the prompt

In [ ]:
# Prerequisites — requires Qdrant running
# docker run -d -p 6333:6333 qdrant/qdrant:latest
from qdrant_client import QdrantClient

QDRANT_HOST = "localhost"
QDRANT_PORT = 6333

try:
    client = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT)
    collections = client.get_collections()
    print(f"✓ Connected to Qdrant at {QDRANT_HOST}:{QDRANT_PORT}")
    print(f"  Existing collections: {[c.name for c in collections.collections]}")
except Exception as e:
    print(f"✗ Cannot reach Qdrant: {e}")
    print(f"  Start it: docker run -d -p 6333:6333 qdrant/qdrant:latest")

## Package Introductions

### qdrant-client
The official Python client for Qdrant vector database. It provides a high-level API for creating collections, inserting vectors, and searching.

Key concepts in Qdrant's data model:
- **Collection** — like a database table. Has a name and a vector configuration (dimensions + distance metric).
- **Point** — like a row. Contains: an ID, a vector (list of floats), and a payload (arbitrary JSON metadata).
- **Payload** — metadata attached to each vector. We store `text`, `page_number`, `filename`, `document_id`, `chunk_index`. This is what lets us show "Source: report.pdf, page 3" in the UI.
- **Search** — given a query vector, find the N closest points by cosine similarity. Returns points sorted by score.

Key APIs:
- `QdrantClient(host, port)` — connect to Qdrant
- `client.create_collection(name, vectors_config)` — create a new collection
- `client.collection_exists(name)` — check if collection exists
- `client.upsert(collection_name, points)` — insert or update points
- `client.search(collection_name, query_vector, limit)` — similarity search
- `client.scroll(collection_name)` — iterate all points (for listing documents)

## Go/TS Comparison

| Concept | Go | Python |
|---------|-----|--------|
| Struct with methods | `type Store struct{} + func (s *Store) Upsert()` | `class QdrantStore: def __init__(), def upsert()` |
| Constructor | `func NewStore() *Store` | `def __init__(self)` |
| Method receiver | `func (s *Store) method()` | `def method(self)` |
| Nil check | `if s.client == nil` | `if self.client is None` |

Python classes are more explicit than Go — every method takes `self` as the first parameter (equivalent to the method receiver in Go). `__init__` is the constructor (runs when you call `MyClass()`). There's no separate `New` function convention.

## Build It

### Step 1: Create the QdrantStore class

This class handles the ingestion side: creating the collection (if it doesn't exist) and upserting vectors with metadata.

In [ ]:
import uuid
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams

class QdrantStore:
    def __init__(self, host: str, port: int, collection_name: str):
        self.client = QdrantClient(host=host, port=port)
        self.collection_name = collection_name
        self._ensure_collection()

    def _ensure_collection(self):
        """Create the collection if it doesn't exist."""
        if not self.client.collection_exists(self.collection_name):
            self.client.create_collection(
                collection_name=self.collection_name,
                vectors_config=VectorParams(
                    size=768,  # nomic-embed-text output dimensions
                    distance=Distance.COSINE,
                ),
            )
            print(f"Created collection: {self.collection_name}")
        else:
            print(f"Collection already exists: {self.collection_name}")

    def upsert(
        self,
        chunks: list[dict],
        vectors: list[list[float]],
        document_id: str,
        filename: str,
    ) -> None:
        """Store chunks with their vectors and metadata."""
        points = [
            PointStruct(
                id=str(uuid.uuid4()),
                vector=vector,
                payload={
                    "text": chunk["text"],
                    "page_number": chunk["page_number"],
                    "chunk_index": chunk["chunk_index"],
                    "document_id": document_id,
                    "filename": filename,
                },
            )
            for chunk, vector in zip(chunks, vectors)
        ]
        self.client.upsert(
            collection_name=self.collection_name,
            points=points,
        )
        print(f"Upserted {len(points)} points for '{filename}'")

    def list_documents(self) -> list[dict]:
        """List all unique documents in the collection."""
        records, _ = self.client.scroll(
            collection_name=self.collection_name,
            limit=10000,
            with_payload=True,
            with_vectors=False,
        )

        docs: dict[str, dict] = {}
        for record in records:
            doc_id = record.payload["document_id"]
            if doc_id not in docs:
                docs[doc_id] = {
                    "document_id": doc_id,
                    "filename": record.payload["filename"],
                    "chunks": 0,
                }
            docs[doc_id]["chunks"] += 1

        return list(docs.values())

# Test it — create a store with a test collection
store = QdrantStore(host="localhost", port=6333, collection_name="lesson_04_test")

### Step 2: Insert some test data

Let's create fake embeddings and upsert them. In the real app, these would come from `embed_texts()`.

In [ ]:
import numpy as np

# Create fake chunks and embeddings for testing
chunks = [
    {"text": "Revenue was 2.5 million dollars.", "page_number": 1, "chunk_index": 0},
    {"text": "Operating margins improved to 23%.", "page_number": 1, "chunk_index": 1},
    {"text": "Engineering team grew to 45 people.", "page_number": 2, "chunk_index": 2},
]

# Fake 768-dimensional vectors (in reality, these come from Ollama)
np.random.seed(42)
vectors = [np.random.randn(768).tolist() for _ in chunks]

store.upsert(
    chunks=chunks,
    vectors=vectors,
    document_id="test-doc-001",
    filename="q3_report.pdf",
)

# List documents
docs = store.list_documents()
for doc in docs:
    print(f"  {doc['filename']}: {doc['chunks']} chunks (ID: {doc['document_id']})")

### Step 3: Build the QdrantRetriever class

This class handles the chat side: searching for similar vectors.

> **Pitfall:** Claude Code sometimes creates the Qdrant collection with the wrong vector size (e.g., 384 or 1536 instead of 768). The size must match your embedding model's output dimensions. nomic-embed-text produces 768-dimensional vectors. If sizes don't match, upsert will fail silently or search will return garbage results.

In [ ]:
class QdrantRetriever:
    def __init__(self, host: str, port: int, collection_name: str):
        self.client = QdrantClient(host=host, port=port)
        self.collection_name = collection_name

    def search(
        self, query_vector: list[float], top_k: int = 5
    ) -> list[dict]:
        """Search for the most similar vectors."""
        results = self.client.search(
            collection_name=self.collection_name,
            query_vector=query_vector,
            limit=top_k,
        )

        return [
            {
                "text": hit.payload["text"],
                "page_number": hit.payload["page_number"],
                "filename": hit.payload["filename"],
                "document_id": hit.payload["document_id"],
                "score": hit.score,
            }
            for hit in results
        ]

# Test search
retriever = QdrantRetriever(host="localhost", port=6333, collection_name="lesson_04_test")

# Search with one of our vectors (should find itself as top result)
results = retriever.search(query_vector=vectors[0], top_k=3)
print("Search results:")
for r in results:
    print(f"  Score: {r['score']:.3f}  Page {r['page_number']}: {r['text']}")

## Experiment

In [ ]:
# Experiment 1: Change top_k
results_1 = retriever.search(query_vector=vectors[0], top_k=1)
results_3 = retriever.search(query_vector=vectors[0], top_k=3)
print(f"top_k=1: {len(results_1)} results")
print(f"top_k=3: {len(results_3)} results")

In [ ]:
# Experiment 2: Search with a random vector (no good matches)
random_vector = np.random.randn(768).tolist()
results = retriever.search(query_vector=random_vector, top_k=3)
print("Search with random vector (expect low scores):")
for r in results:
    print(f"  Score: {r['score']:.3f}  {r['text']}")

In [ ]:
# Clean up — delete test collection
store.client.delete_collection("lesson_04_test")
print("Cleaned up test collection")

## Check Your Understanding

1. **Why use a vector database instead of storing embeddings in Postgres with pgvector?** (Hint: think about search speed at scale and purpose-built indexing)

2. **What's stored in the Qdrant payload, and why? What would happen if we only stored the vector without metadata?**

3. **In the `QdrantStore.__init__`, we call `_ensure_collection()`. What's the Go equivalent of this initialization pattern?** (Hint: think about `sync.Once` or init functions)